
#  Smart Energy Load Forecasting and Management in Microgrid Using LSTM Neural Networks





 STEP 0 : Install / Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')



 STEP 1 : Load & Convert Dataset


In [ ]:
print("\n[1/7] Loading Dataset...")

# TXT file load karo directly
df_raw = pd.read_csv('/content/household_power_consumption.txt',
                     sep=';',
                     parse_dates={'datetime': ['Date', 'Time']},
                     infer_datetime_format=True,
                     na_values=['?'],
                     low_memory=False)

# CSV mein save karo (optional but good practice)
df_raw.to_csv('/content/power_data.csv', index=False)
print("   TXT → CSV conversion done!")

# Index set karo
df_raw.set_index('datetime', inplace=True)

print(f"   Raw data shape   : {df_raw.shape}")
print(f"   Date range       : {df_raw.index.min()} → {df_raw.index.max()}")
print(f"   Columns          : {list(df_raw.columns)}")


[1/7] Loading Dataset...
   TXT → CSV conversion done!
   Raw data shape   : (2075259, 7)
   Date range       : 2006-12-16 17:24:00 → 2010-11-26 21:02:00
   Columns          : ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


 STEP 2 : Preprocessing


In [ ]:
print("\n[2/7] Preprocessing...")

# Use Global_active_power as load (kW)
df = df_raw[['Global_active_power']].copy()
df.columns = ['load_kw']

# Resample minute data → hourly average
df = df.resample('h').mean()

# Fill missing values with forward fill
df.fillna(method='ffill', inplace=True)
df.dropna(inplace=True)

print(f"   Hourly samples   : {len(df)}")
print(f"   Missing values   : {df.isnull().sum().values[0]}")
print(f"   Min load         : {df['load_kw'].min():.3f} kW")
print(f"   Max load         : {df['load_kw'].max():.3f} kW")
print(f"   Mean load        : {df['load_kw'].mean():.3f} kW")


[2/7] Preprocessing...
   Hourly samples   : 34589
   Missing values   : 0
   Min load         : 0.124 kW
   Max load         : 6.561 kW
   Mean load        : 1.089 kW



STEP 3 : Simulate Solar Generation Data
         (sine-curve based — realistic)


In [ ]:
print("\n[3/7] Simulating Solar Generation Data...")

hours = np.arange(len(df))
hour_of_day = df.index.hour

# Solar generation peaks at noon (hour 12), zero at night
solar_base = np.maximum(0, np.sin((hour_of_day - 6) * np.pi / 12))

# Add seasonal variation (summer more, winter less)
month = df.index.month
seasonal = 0.7 + 0.3 * np.sin((month - 3) * np.pi / 6)

# Add random cloud noise
np.random.seed(42)
noise = np.random.uniform(0.85, 1.15, size=len(df))

# Max solar capacity = 2.5 kW (typical small microgrid)
df['solar_kw'] = solar_base * seasonal * noise * 2.5
df['solar_kw'] = df['solar_kw'].clip(lower=0)

# Battery capacity simulation (simplified)
BATTERY_CAPACITY = 5.0  # kWh
df['net_load'] = df['load_kw'] - df['solar_kw']

print(f"   Avg solar gen    : {df['solar_kw'].mean():.3f} kW")
print(f"   Max solar gen    : {df['solar_kw'].max():.3f} kW")


[3/7] Simulating Solar Generation Data...
   Avg solar gen    : 0.558 kW
   Max solar gen    : 2.874 kW


STEP 4 : Create Sequences for LSTM

In [ ]:
print("\n[4/7] Creating LSTM Sequences...")

WINDOW_SIZE = 24   # past 24 hours as input
FORECAST    = 4    # predict next 4 hours

# Normalize load data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_load = scaler.fit_transform(df[['load_kw']])

def create_sequences(data, window, forecast):
    X, y = [], []
    for i in range(len(data) - window - forecast + 1):
        X.append(data[i : i + window])
        y.append(data[i + window : i + window + forecast].flatten())
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_load, WINDOW_SIZE, FORECAST)

# Train / Test split (80 / 20)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"   Window size      : {WINDOW_SIZE} hours")
print(f"   Forecast horizon : {FORECAST} hours")
print(f"   Training samples : {X_train.shape[0]}")
print(f"   Testing  samples : {X_test.shape[0]}")
print(f"   X shape          : {X_train.shape}  →  (samples, timesteps, features)")


[4/7] Creating LSTM Sequences...
   Window size      : 24 hours
   Forecast horizon : 4 hours
   Training samples : 27649
   Testing  samples : 6913
   X shape          : (27649, 24, 1)  →  (samples, timesteps, features)


STEP 5 : Build & Train LSTM Model

In [ ]:
print("\n[5/7] Building LSTM Model...")

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(FORECAST)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\n   Training in progress...")
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)
print("   Training complete!")


[5/7] Building LSTM Model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,444 (115.02 KB)

 Trainable params: 29,444 (115.02 KB)

 Non-trainable params: 0 (0.00 B)


   Training in progress...
Epoch 1/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 16s 33ms/step - loss: 0.0189 - mae: 0.1058 - val_loss: 0.0186 - val_mae: 0.1092
Epoch 2/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 13s 32ms/step - loss: 0.0162 - mae: 0.0965 - val_loss: 0.0162 - val_mae: 0.0966
Epoch 3/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 20s 30ms/step - loss: 0.0143 - mae: 0.0885 - val_loss: 0.0145 - val_mae: 0.0910
Epoch 4/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - loss: 0.0135 - mae: 0.0850 - val_loss: 0.0140 - val_mae: 0.0888
Epoch 5/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 13s 33ms/step - loss: 0.0132 - mae: 0.0837 - val_loss: 0.0138 - val_mae: 0.0880
Epoch 6/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - loss: 0.0131 - mae: 0.0830 - val_loss: 0.0137 - val_mae: 0.0898
Epoch 7/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - loss: 0.0129 - mae: 0.0823 - val_loss: 0.0138 - val_mae: 0.0863
Epoch 8/30
389/389 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - loss: 0.0128 - mae: 0.0820 - val_loss: 0.0136 - val_mae: 0.0864
Epoch 9/30
3

STEP 6 : Evaluate Model

In [ ]:
print("\n[6/7] Evaluating Model...")

y_pred_scaled = model.predict(X_test)

# Inverse transform (back to kW)
def inverse_multi(scaled, scaler):
    results = []
    for row in scaled:
        dummy = np.zeros((len(row), 1))
        dummy[:, 0] = row
        results.append(scaler.inverse_transform(dummy).flatten())
    return np.array(results)

y_pred_kw = inverse_multi(y_pred_scaled, scaler)
y_test_kw = inverse_multi(y_test,        scaler)

# Metrics (averaged across 4-hour forecast)
rmse = np.sqrt(mean_squared_error(y_test_kw.flatten(), y_pred_kw.flatten()))
mae  = mean_absolute_error(y_test_kw.flatten(), y_pred_kw.flatten())
def smape(y_true, y_pred):
    return 100 * np.mean(
        np.abs(y_pred - y_true) /
        ((np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8)
    )

smape_value = smape(y_test_kw.flatten(), y_pred_kw.flatten())
print(f"SMAPE : {smape_value:.2f} %")

print(f"\n   ┌─────────────────────────────┐")
print(f"   │  Model Evaluation Results   │")
print(f"   ├─────────────────────────────┤")
print(f"   │  RMSE  : {rmse:.4f} kW           │")
print(f"   │  MAE   : {mae:.4f} kW           │")
print(f"   │  SMAPE  : {smape_value:.2f} %             │")
print(f"   └─────────────────────────────┘")


[6/7] Evaluating Model...
217/217 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
SMAPE : 44.32 %

   ┌─────────────────────────────┐
   │  Model Evaluation Results   │
   ├─────────────────────────────┤
   │  RMSE  : 0.5829 kW           │
   │  MAE   : 0.4252 kW           │
   │  SMAPE  : 44.32 %             │
   └─────────────────────────────┘


STEP 7 : Smart Microgrid Suggestions

In [ ]:
print("\n[7/7] Generating Smart Microgrid Suggestions...")

def get_microgrid_suggestion(forecast_kw, solar_kw, battery_level_pct):
    """
    Rule-based microgrid energy management engine.
    Inputs:
        forecast_kw       : predicted load for next hour (kW)
        solar_kw          : current solar generation (kW)
        battery_level_pct : current battery % (0 to 100)
    Returns:
        action  : recommended action string
        source  : energy source to use
        saving  : estimated saving in kWh
    """
    net = forecast_kw - solar_kw
    saving = 0.0

    if forecast_kw < 0.5:
        action = "Very low load — store excess solar in battery"
        source = "Solar + Charge Battery 🔋"
        saving = solar_kw * 0.9

    elif solar_kw >= forecast_kw:
        action = "Solar is sufficient — switch to solar, disconnect grid"
        source = "Solar Only 🌞"
        saving = forecast_kw

    elif solar_kw > 0.5 and battery_level_pct > 30:
        action = "Moderate load — use solar + battery combination"
        source = "Solar + Battery ⚡"
        saving = solar_kw + (battery_level_pct / 100) * BATTERY_CAPACITY * 0.3

    elif solar_kw > 0.5 and battery_level_pct <= 30:
        action = "Battery low — use solar + import from grid"
        source = "Solar + Grid 🔌"
        saving = solar_kw * 0.5

    elif forecast_kw > 2.0 and battery_level_pct > 50:
        action = "High load peak — use grid + discharge battery"
        source = "Grid + Battery ⚡🔌"
        saving = (battery_level_pct / 100) * BATTERY_CAPACITY * 0.2

    elif forecast_kw > 2.5:
        action = "⚠️  ALERT: Very high load — consider load shedding"
        source = "Grid (Reduce Non-essential Load) ⚠️"
        saving = 0.0

    else:
        action = "Normal load — grid supply sufficient"
        source = "Main Grid 🔌"
        saving = 0.0

    return action, source, round(saving, 3)

# Demo: run suggestions for last 10 test predictions
print("\n   Sample Microgrid Suggestions (last 10 test hours):\n")
print(f"   {'Hour':<6} {'Forecast(kW)':<15} {'Solar(kW)':<12} {'Action':<50} {'Source':<30} {'Saving(kWh)'}")
print("   " + "─" * 130)

sample_solar  = df['solar_kw'].values[-len(y_test):]
battery_level = 60  # assume 60% charged at start

for i in range(-10, 0):
    fc_kw   = y_pred_kw[i][0]
    sol_kw  = sample_solar[i]
    action, source, saving = get_microgrid_suggestion(fc_kw, sol_kw, battery_level)
    print(f"   {i+10:<6} {fc_kw:<15.3f} {sol_kw:<12.3f} {action:<50} {source:<30} {saving}")


# ─────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────
print("\n Generating Plots...")

fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle('Smart Energy Load Forecasting & Management in Microgrids\nUsing LSTM Neural Networks',
             fontsize=14, fontweight='bold', y=0.98)

# ── Plot 1: Training Loss ──
ax1 = axes[0]
ax1.plot(history.history['loss'],     label='Training Loss',   color='steelblue',  linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation Loss', color='darkorange', linewidth=2, linestyle='--')
ax1.set_title('LSTM Model Training Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss (MSE)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Plot 2: Predicted vs Actual (first 200 test samples, hour-0 forecast) ──
ax2 = axes[1]
n_show = min(200, len(y_test_kw))
ax2.plot(y_test_kw[:n_show, 0], label='Actual Load (kW)',    color='steelblue',  linewidth=1.5)
ax2.plot(y_pred_kw[:n_show, 0], label='Predicted Load (kW)', color='crimson',    linewidth=1.5, linestyle='--')
ax2.fill_between(range(n_show),
                 y_test_kw[:n_show, 0],
                 y_pred_kw[:n_show, 0],
                 alpha=0.15, color='crimson', label='Error')
ax2.set_title(f'Actual vs Predicted Energy Load  |  RMSE={rmse:.4f} kW  MAE={mae:.4f} kW  SMAPE={smape_value:.2f}%',
              fontsize=11, fontweight='bold')
ax2.set_xlabel('Test Samples (hours)')
ax2.set_ylabel('Load (kW)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# ── Plot 3: Load vs Solar (one week sample) ──
ax3 = axes[2]
week_df = df.iloc[:168]   # first 7 days = 168 hours
ax3.fill_between(week_df.index, week_df['load_kw'],  alpha=0.6, color='steelblue',  label='Load (kW)')
ax3.fill_between(week_df.index, week_df['solar_kw'], alpha=0.6, color='goldenrod',  label='Solar Generation (kW)')
ax3.plot(week_df.index, week_df['net_load'].clip(lower=0),
         color='crimson', linewidth=1.5, linestyle='--', label='Net Load from Grid (kW)')
ax3.set_title('Load vs Solar Generation — 1 Week Overview (Microgrid View)',
              fontsize=11, fontweight='bold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Power (kW)')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('microgrid_lstm_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("   Plot saved as: microgrid_lstm_results.png")


# ─────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  PROJECT SUMMARY")
print("=" * 60)
print(f"  Title    : Smart Energy Load Forecasting and")
print(f"             Management in Microgrids using LSTM")
print(f"  Dataset  : UCI Household Power Consumption")
print(f"             + Simulated Solar Generation")
print(f"  Model    : Stacked LSTM (64 → 32 → Dense)")
print(f"  Window   : {WINDOW_SIZE} hours input → {FORECAST} hours forecast")
print(f"  RMSE     : {rmse:.4f} kW")
print(f"  MAE      : {mae:.4f} kW")
print(f"  SMAPE     : {smape_value:.2f} %")
print(f"  Output   : Load Forecast + Microgrid Action Suggestions")
print("=" * 60)




[7/7] Generating Smart Microgrid Suggestions...

   Sample Microgrid Suggestions (last 10 test hours):

   Hour   Forecast(kW)    Solar(kW)    Action                                             Source                         Saving(kWh)
   ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


NameError: name 'df' is not defined

In [ ]:

#  GRADIO UI — Day + Time based prediction


import gradio as gr
import matplotlib
matplotlib.use('Agg')

print("Preparing UI...")


# Helper — get typical 24hr pattern for

def get_typical_pattern(day_of_week, hour):
    """
    day_of_week : 0=Monday ... 6=Sunday
    hour        : 0 to 23
    Returns last 24 hours typical load pattern
    """
    # Filter dataset for matching day + hour
    mask = (df.index.dayofweek == day_of_week) & (df.index.hour == hour)
    matching = df[mask]

    if len(matching) == 0:
        # Fallback — use just hour match
        mask = df.index.hour == hour
        matching = df[mask]

    # Pick the most typical occurrence (closest to median)
    median_val = matching['load_kw'].median()
    closest_idx = (matching['load_kw'] - median_val).abs().idxmin()
    position = df.index.get_loc(closest_idx)

    # Make sure we have 24 hours before this point
    if position < WINDOW_SIZE:
        position = WINDOW_SIZE

    return position


# ─────────────────────────────────────────
# Suggestion Engine
# ─────────────────────────────────────────
def get_suggestion_ui(forecast_kw, solar_kw, battery_pct):
    if forecast_kw < 0.5:
        return "🔋 Very Low Load — Store excess solar in battery"
    elif solar_kw >= forecast_kw:
        return "🌞 Solar Sufficient — Switch to solar, disconnect grid!"
    elif solar_kw > 0.5 and battery_pct > 30:
        return "⚡ Moderate Load — Use Solar + Battery combination"
    elif solar_kw > 0.5 and battery_pct <= 30:
        return "🔌 Battery Low — Use Solar + import from Grid"
    elif forecast_kw > 2.0 and battery_pct > 50:
        return "⚡🔌 High Load Peak — Use Grid + discharge Battery"
    elif forecast_kw > 2.5:
        return "⚠️ ALERT: Very High Load — Reduce non-essential load!"
    else:
        return "🔌 Normal Load — Grid supply sufficient"


# ─────────────────────────────────────────
# Main Prediction Function
# ─────────────────────────────────────────
def predict_energy(day_name, hour_str, battery_level, season):

    try:
        # Map inputs
        day_map = {
            "Monday": 0, "Tuesday": 1, "Wednesday": 2,
            "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6
        }
        season_month_map = {
            "Summer (Apr–Jun)": 5,
            "Monsoon (Jul–Sep)": 8,
            "Winter (Oct–Dec)": 11,
            "Spring (Jan–Mar)": 2
        }

        day_of_week = day_map[day_name]
        hour        = int(hour_str.replace(":00", ""))
        month       = season_month_map[season]

        # Get typical pattern position from dataset
        position = get_typical_pattern(day_of_week, hour)

        # Extract past 24 hours
        past_24    = df['load_kw'].values[position - WINDOW_SIZE : position]
        past_hours = [f"{(hour - WINDOW_SIZE + i) % 24:02d}:00"
                      for i in range(WINDOW_SIZE)]

        # Scale + Predict
        past_scaled = scaler.transform(past_24.reshape(-1, 1))
        X_input     = past_scaled.reshape(1, WINDOW_SIZE, 1)
        pred_scaled = model.predict(X_input, verbose=0)
        pred_kw     = scaler.inverse_transform(
                          pred_scaled.reshape(-1, 1)
                      ).flatten()

        # Solar for next 4 hours
        future_hours = [(hour + i + 1) % 24 for i in range(FORECAST)]
        future_solar = []
        for h in future_hours:
            s   = max(0, np.sin((h - 6) * np.pi / 12))
            sea = 0.7 + 0.3 * np.sin((month - 3) * np.pi / 6)
            future_solar.append(round(s * sea * 2.5, 3))

        # ── Plot ──
        fig, axes = plt.subplots(2, 1, figsize=(11, 8))
        fig.patch.set_facecolor('#0f1117')
        fig.suptitle(
            f'Energy Forecast — {day_name}, {hour:02d}:00  |  {season}',
            fontsize=13, fontweight='bold', color='white'
        )
        for ax in axes:
            ax.set_facecolor('#1a1a2e')
            ax.tick_params(colors='white')
            ax.xaxis.label.set_color('white')
            ax.yaxis.label.set_color('white')
            ax.title.set_color('white')
            for spine in ax.spines.values():
                spine.set_edgecolor('#444')

        # Plot 1 — Past 24 hours typical pattern
        axes[0].plot(range(WINDOW_SIZE), past_24,
                     color='#00d4ff', linewidth=2,
                     label=f'Typical Load Pattern — {day_name}')
        axes[0].fill_between(range(WINDOW_SIZE), past_24,
                             alpha=0.2, color='#00d4ff')
        axes[0].set_xticks(range(0, WINDOW_SIZE, 2))
        axes[0].set_xticklabels(
            [past_hours[i] for i in range(0, WINDOW_SIZE, 2)],
            rotation=45, color='white', fontsize=8
        )
        axes[0].set_title(
            f'Past 24 Hours — Typical {day_name} Load Pattern',
            fontsize=11, fontweight='bold'
        )
        axes[0].set_ylabel('Load (kW)', color='white')
        axes[0].legend(facecolor='#1a1a2e', labelcolor='white')
        axes[0].grid(True, alpha=0.2)

        # Plot 2 — Next 4 hours forecast vs solar
        x_labels = [f"+{i+1}hr\n{future_hours[i]:02d}:00"
                    for i in range(FORECAST)]
        x     = np.arange(FORECAST)
        width = 0.35

        bars1 = axes[1].bar(x - width/2, pred_kw,      width,
                            color='#ff6b6b', alpha=0.9,
                            label='Predicted Load (kW)')
        bars2 = axes[1].bar(x + width/2, future_solar, width,
                            color='#ffd93d', alpha=0.9,
                            label='Solar Available (kW)')

        # Value labels on bars
        for bar in bars1:
            axes[1].text(bar.get_x() + bar.get_width()/2,
                         bar.get_height() + 0.03,
                         f'{bar.get_height():.2f}',
                         ha='center', va='bottom',
                         color='white', fontsize=9)
        for bar in bars2:
            axes[1].text(bar.get_x() + bar.get_width()/2,
                         bar.get_height() + 0.03,
                         f'{bar.get_height():.2f}',
                         ha='center', va='bottom',
                         color='white', fontsize=9)

        axes[1].set_xticks(x)
        axes[1].set_xticklabels(x_labels, color='white')
        axes[1].set_title(
            'Next 4 Hours — Predicted Load vs Solar Generation',
            fontsize=11, fontweight='bold'
        )
        axes[1].set_ylabel('Power (kW)', color='white')
        axes[1].legend(facecolor='#1a1a2e', labelcolor='white')
        axes[1].grid(True, alpha=0.2, axis='y')

        plt.tight_layout(pad=2.5)

        # ── Forecast Text ──
        forecast_text = ""
        for i in range(FORECAST):
            forecast_text += (
                f"🕐 {future_hours[i]:02d}:00"
                f"  →  Load: {pred_kw[i]:.3f} kW"
                f"  |  Solar: {future_solar[i]:.3f} kW\n"
            )

        # ── Suggestion Text ──
        suggestion_text = ""
        for i in range(FORECAST):
            sug = get_suggestion_ui(pred_kw[i], future_solar[i], battery_level)
            suggestion_text += f"  {future_hours[i]:02d}:00  →  {sug}\n"

        # ── Summary ──
        total_load   = sum(pred_kw)
        total_solar  = sum(future_solar)
        energy_saved = min(total_solar, total_load)
        saving_pct   = (energy_saved / total_load * 100) if total_load > 0 else 0

        grid_needed  = max(0, total_load - total_solar)

        summary = (
            f"📊 Total Predicted Load      : {total_load:.3f} kWh\n"
            f"☀️  Total Solar Available    : {total_solar:.3f} kWh\n"
            f"💰 Energy Saved (Solar)      : {energy_saved:.3f} kWh  ({saving_pct:.1f}%)\n"
            f"🔌 Grid Import Needed        : {grid_needed:.3f} kWh\n"
            f"🔋 Battery Level             : {battery_level}%\n"
            f"📅 Scenario                  : {day_name} at {hour:02d}:00  |  {season}"
        )

        status = f"✅ Prediction done for {day_name} {hour:02d}:00!"
        return fig, status, forecast_text, suggestion_text, summary

    except Exception as e:
        return None, f"❌ Error: {str(e)}", "", "", ""


# ─────────────────────────────────────────
# Gradio UI
# ─────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Base(primary_hue="blue", secondary_hue="cyan"),
    title="Smart Microgrid Energy Forecasting"
) as app:


    # Inputs
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎛️ Select Scenario")

            day_input = gr.Dropdown(
                choices=["Monday", "Tuesday", "Wednesday",
                         "Thursday", "Friday", "Saturday", "Sunday"],
                value="Monday",
                label="📅 Day of Week",
                info="Which day do you want to forecast for?"
            )
            hour_input = gr.Dropdown(
                choices=[f"{h:02d}:00" for h in range(24)],
                value="09:00",
                label="🕐 Time of Day",
                info="What time do you want to start forecasting from?"
            )
            season_input = gr.Dropdown(
                choices=["Summer (Apr–Jun)", "Monsoon (Jul–Sep)",
                         "Winter (Oct–Dec)", "Spring (Jan–Mar)"],
                value="Summer (Apr–Jun)",
                label="🌤️ Season",
                info="Season affects solar generation intensity"
            )
            battery_input = gr.Slider(
                minimum=0, maximum=100, value=60, step=5,
                label="🔋 Current Battery Level (%)",
                info="How much battery is currently charged?"
            )
            predict_btn = gr.Button(
                "🔮 Predict & Get Suggestions",
                variant="primary", size="lg"
            )



    gr.Markdown("---")

    # Status
    status_out = gr.Textbox(label="Status", interactive=False, max_lines=1)

    # Outputs
    with gr.Row():
        with gr.Column(scale=2):
            plot_out = gr.Plot(label="📈 Energy Forecast Chart")

        with gr.Column(scale=1):
            gr.Markdown("### 🕐 Hour-wise Forecast")
            forecast_out = gr.Textbox(
                label="Predicted Load & Solar (next 4 hours)",
                lines=6, interactive=False
            )
            gr.Markdown("### 🎯 Microgrid Suggestions")
            suggestion_out = gr.Textbox(
                label="Smart Energy Switching Recommendations",
                lines=6, interactive=False
            )

    gr.Markdown("### 📊 Energy Summary")
    summary_out = gr.Textbox(label="Summary", lines=7, interactive=False)



    # Button
    predict_btn.click(
        fn=predict_energy,
        inputs=[day_input, hour_input, battery_input, season_input],
        outputs=[plot_out, status_out, forecast_out, suggestion_out, summary_out]
    )

print("Launching Gradio UI...")
app.launch(share=True, debug=False)

Preparing UI...
Launching Gradio UI...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7efa8fefc89194da89.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
